In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from amazonas_pipeline.utils import generate_boxes, add_pop_and_smod_to_cells, get_area_by_smod_from_polys
import os

In [2]:
ghsl_path = Path(os.environ["GHSL_PATH"])

df_final_path = Path("./generated/df_final")
cells_path = Path("./generated/cells")

df_final_fixed_path = Path("./generated/df_final_fixed")
df_final_fixed_path.mkdir(exist_ok=True)

In [3]:
year = 1975

for year in range(1975, 2021, 5):
    df_final = gpd.read_file(df_final_path / f"{year}.gpkg")
    df_cells = gpd.read_file(cells_path / f"{year}.gpkg")

    df_final = (
        df_final.drop(
            columns=["area_rural_km2", "area_urban_cluster_km2", "area_urban_center_km2"],
            errors="ignore",
        )
        .join(get_area_by_smod_from_polys(df_final, df_cells, year=year))
        .assign(
            calculated_area=lambda df: (
                df["area_rural_km2"]
                + df["area_urban_cluster_km2"]
                + df["area_urban_center_km2"]
            ),
        )
    )

    right_mask = df_final["area_km2"] == df_final["calculated_area"]
    right_polys = df_final.loc[right_mask].drop(columns=["calculated_area"])
    wrong_polys = df_final.loc[~right_mask].drop(columns=["area_rural_km2", "area_urban_cluster_km2", "area_urban_center_km2", "calculated_area"])

    crs = df_final.crs
    if crs is None:
        err = "CRS is not defined in df_final. Please provide a valid CRS."
        raise ValueError(err)

    df_boxes_all = []
    for _, row in wrong_polys.bounds.iterrows():
        boxes = add_pop_and_smod_to_cells(generate_boxes(row["minx"], row["miny"], row["maxx"], row["maxy"], crs=crs), selected_year=year, ghsl_path=ghsl_path)
        df_boxes_all.append(boxes)

    df_boxes_all = pd.concat(df_boxes_all, ignore_index=True).reset_index(names="cell_id")

    wrong_polys = wrong_polys.join(get_area_by_smod_from_polys(wrong_polys, df_boxes_all, year=year))

    df_final_fixed = pd.concat([right_polys, wrong_polys]).sort_index().pipe(lambda df: gpd.GeoDataFrame(df, geometry="geometry", crs=crs))
    df_final_fixed.to_file(df_final_fixed_path / f"{year}.gpkg", driver="GPKG")

KeyboardInterrupt: 